In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

# [숫자 노트북(1_1_10)과의 차이] 입력이 숫자가 아니라 "영문 대소문자"라서 문자 → 숫자 변환 단계가 추가된다.
# cal_len : 시퀀스 길이를 변수로 뺐다. 5를 10으로 바꾸면 "10글자 역순"을 같은 코드로 돌려 볼 수 있다
#           (파일명의 "5_10비교"가 이 뜻. 길이를 바꿀 때 아래 모든 5가 이 변수로 연결돼 있음)
cal_len = 5
# 문자와 인덱스 매핑
# chars : 사용할 문자 전체 = 대문자 26개 + 소문자 26개 = 52종류 (숫자 노트북의 0~9 열 가지에 해당)
#         대소문자는 서로 다른 문자로 취급한다 ('A' ≠ 'a').
chars = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
# num_classes : 분류해야 할 클래스 수 = 52. 아래 Embedding 입력 크기와 Dense 출력 크기에 쓰인다 (숫자 노트북에선 10)
num_classes = len(chars)

# 문자 ↔ 숫자 매핑 사전. 신경망은 문자를 직접 못 다루므로 'A'→0, 'B'→1, ..., 'Z'→25, 'a'→26, ..., 'z'→51 로 바꾼다.
#  - char_to_index : 입력/정답 문자를 번호로 바꿀 때 사용 (인코딩)
#  - index_to_char : 모델이 낸 번호를 다시 문자로 되돌릴 때 사용 (디코딩, 맨 아래 예측 부분에서 사용)
char_to_index = {char: idx for idx, char in enumerate(chars)}
index_to_char = {idx: char for idx, char in enumerate(chars)}

# 학습 데이터 생성
# np.random.choice(list(chars), size=(10000, cal_len)) : 52개 문자 중에서 무작위로 (10000, 5)개를 뽑는다.
#  - 숫자 노트북의 np.random.randint와 달리 결과가 "숫자 배열"이 아니라 "문자(문자열) 배열"이다 (dtype '<U1').
#  - 같은 문자가 여러 번 나올 수 있다 (복원 추출이 기본값).
#  - 가능한 조합 수는 52^5 ≈ 3.8억 가지다. 숫자 노트북은 10^5 = 10만 가지라서, 학습에 쓰는 1만 개가 차지하는 비율이 훨씬 작다.
train_data = np.random.choice(list(chars), size=(10000, cal_len))
# 정답 = 입력을 열 방향으로 뒤집은 것 (숫자 노트북과 동일). 아직 문자 상태다.
label_data = np.flip(train_data, axis=1)

# 문자 데이터를 숫자 인덱스로 변환
# 문자 → 번호 변환 (이중 리스트 컴프리헨션: 바깥은 샘플 10000개, 안쪽은 샘플 하나의 글자 5개).
#  [['N','D','j','c','W'], ...] → [[13, 3, 35, 28, 22], ...]  결과는 정수 ndarray (10000, 5)
#  입력뿐 아니라 "정답"도 번호로 바꿔야 sparse_categorical_crossentropy가 쓸 수 있다
#  (숫자 노트북은 처음부터 숫자라서 이 단계가 필요 없었음).
train_data_indices = np.array([[char_to_index[char] for char in seq] for seq in train_data])
label_data_indices = np.array([[char_to_index[char] for char in seq] for seq in label_data])

# 문자 상태 그대로 앞 4개를 찍어 입력과 정답이 역순인지 확인 (번호가 아니라 문자로 보여서 눈으로 확인하기 쉽다)
print(train_data[:4])
print(label_data[:4])

# pad_sequence 전처리
# pad_sequences : 문자열이 아니라 "번호로 바꾼 데이터"에 적용한다. 모두 길이 cal_len으로 같아서 값은 그대로다.
#  - 숫자 노트북은 정답(label_data)에는 적용하지 않았는데, 여기서는 정답에도 같은 처리를 해서 padded_labels를 만든다.
#  - 번호 0은 'A'인데 패딩 값도 0이다. 패딩이 실제로 일어나지 않아 지금은 문제가 없다.
padded_train = pad_sequences(train_data_indices, maxlen=cal_len)
padded_labels = pad_sequences(label_data_indices, maxlen=cal_len)

[['N' 'D' 'j' 'c' 'W']
 ['h' 'v' 'S' 'm' 'i']
 ['K' 'J' 'W' 'i' 'c']
 ['v' 'F' 'q' 'c' 'Q']]
[['W' 'c' 'j' 'D' 'N']
 ['i' 'm' 'S' 'v' 'h']
 ['c' 'i' 'W' 'J' 'K']
 ['Q' 'c' 'q' 'F' 'v']]


In [3]:
# 번호로 바뀐 입력 데이터 (10000, 5) 확인
padded_train

array([[13,  3, 35, 28, 22],
       [33, 47, 18, 38, 34],
       [10,  9, 22, 34, 28],
       ...,
       [44, 26, 44, 40, 15],
       [ 9,  3, 27,  4, 44],
       [20, 10, 37, 42, 38]], dtype=int32)

In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

# (아래 "데이터 준비" 부분은 위 0번 셀과 같은 코드다. 문자→번호 변환 설명은 0번 셀 주석 참고)
cal_len = 5
# 문자와 인덱스 매핑
chars = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
num_classes = len(chars)

char_to_index = {char: idx for idx, char in enumerate(chars)}
index_to_char = {idx: char for idx, char in enumerate(chars)}

# 학습 데이터 생성
train_data = np.random.choice(list(chars), size=(10000, cal_len))
label_data = np.flip(train_data, axis=1)

# 문자 데이터를 숫자 인덱스로 변환
train_data_indices = np.array([[char_to_index[char] for char in seq] for seq in train_data])
label_data_indices = np.array([[char_to_index[char] for char in seq] for seq in label_data])

print(train_data[:4])
print(label_data[:4])

# pad_sequence 전처리
padded_train = pad_sequences(train_data_indices, maxlen=cal_len)
padded_labels = pad_sequences(label_data_indices, maxlen=cal_len)

# 모델링
# Sequential 틀과 층 구성은 숫자 노트북(1_1_10)과 같은 구조: Embedding → Bidirectional LSTM → Dense(softmax)
model = Sequential()
# [숫자 노트북과의 차이] 하드코딩된 10과 5 대신 변수를 쓴다.
#  - Embedding(num_classes=52, 100) : 문자 52종류 각각을 100차원 벡터로 바꾼다. 파라미터 52 x 100 = 5,200 (숫자 노트북은 1,000).
#  - input_length=cal_len : 입력 길이. 경고 메시지 "input_length is deprecated"가 출력에 보이는데,
#    Keras 3에서 없어도 되는 인자라는 뜻이고 학습에는 영향이 없다.
model.add(Embedding(num_classes, 100, input_length=cal_len))
# 숫자 노트북과 완전히 같다 (양방향 LSTM 64, 모든 위치의 출력 유지). 출력 (샘플 수, 5, 128), 파라미터 84,480.
# [참고] 파일명의 "OnlyLSTM"은 인코더-디코더(seq2seq) 구조 없이 LSTM 한 층만 쓴다는 뜻이다.
#        입력과 출력 길이가 같아서 "각 위치의 글자 맞히기"로 풀리기 때문에 가능하다.
model.add(Bidirectional(LSTM(64, return_sequences=True)))
# [숫자 노트북과의 차이] 출력 뉴런이 10개 → 52개. 위치마다 52종류 문자 중 하나를 고르는 분류 문제가 됐다.
#  - 파라미터 128 x 52 + 52 = 6,708 (숫자 노트북은 1,290)
#  - 전체 파라미터 = 5,200 + 84,480 + 6,708 = 96,388  (model.summary()의 Total params와 비교)
#  - 찍기만 해도 맞힐 확률이 1/52 ≈ 1.9%라서(숫자는 1/10) 더 어려운 문제다.
model.add(Dense(num_classes, activation='softmax'))

# 모델의 요약을 보기 위해 한 번 모델을 호출하여 초기화합니다
# 입력 (샘플 수 미정, 길이 cal_len)로 모델을 미리 만들어 summary()를 볼 수 있게 한다
model.build(input_shape=(None, cal_len))
model.summary()

# 모델 컴파일
# 손실/최적화/정확도 지표는 숫자 노트북과 같다 (정답이 정수 번호이므로 sparse_categorical_crossentropy)
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# ------------------------------------------------------------------
# [숫자 노트북과의 차이] EarlyStopping(조기 종료)을 사용한다.
#  - 숫자 노트북은 epochs=10으로 고정이었지만, 여기는 500까지 허용하되 더 나아지지 않으면 멈춘다.
#  - monitor='val_loss' : 검증 데이터(훈련 데이터의 마지막 20%)의 손실을 지켜본다.
#  - patience=10 : val_loss가 10 epoch 연속 개선되지 않으면 학습을 중단한다.
#  - restore_best_weights=True : 중단 시점이 아니라 val_loss가 가장 좋았던 epoch의 가중치로 되돌려 놓는다.
#  - 문자 52종류는 어려워서 학습이 오래 걸리기 때문에, 몇 epoch가 필요한지 미리 정하지 않고 자동으로 멈추게 했다.
# ------------------------------------------------------------------
from tensorflow.keras.callbacks import EarlyStopping

# 조기 종료 콜백
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# 모델 학습
# fit : 입력은 padded_train, 정답은 padded_labels(둘 다 번호로 바뀐 (10000, 5) 정수 배열).
#  - epochs=500 : 상한선일 뿐, 실제로는 EarlyStopping이 훨씬 일찍 멈춘다.
#  - validation_split=0.2 : 앞 80%(8000개)로 학습, 뒤 20%(2000개)로 검증. 데이터가 이미 랜덤이라 섞지 않아도 된다.
#  - callbacks=[early_stop] : 위에서 만든 조기 종료 규칙을 적용
history = model.fit(
    padded_train, padded_labels,
    epochs=500,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]  # ← 여기에 추가!
)

# loss와 accuracy
# evaluate : 학습에 쓴 데이터 전체(10000개)로 다시 평가한 값이라 "새 데이터에 대한 성능"이 아니다.
#  - acc는 "위치별" 정확도다 (글자 5개 중 몇 개를 맞혔는지의 평균). 아래의 "총 정확도"와 기준이 다르다.
loss, acc = model.evaluate(padded_train, padded_labels)
print(f"loss : {loss:.4f}, acc = {acc:.4f}")

# 랜덤한 영문 대소문자 5글자를 요소로 하는 20개 리스트 생성
# ------------------------------------------------------------------
# [숫자 노트북과의 차이] 직접 입력(input) 대신, 새로 만든 20개 문자열로 자동 검증한다.
#  - 학습 때 쓰지 않은 새 랜덤 문자열 20개를 만든다. 문자 5개를 이어 붙인 문자열이다. 예: 'kMIZi'
# ------------------------------------------------------------------
test_data = ["".join(np.random.choice(list(chars), cal_len)) for _ in range(20)]
# 거꾸로 된 리스트 생성
# 정답 문자열 = 입력 문자열을 뒤집은 것. 파이썬 슬라이싱 [::-1]은 문자열을 거꾸로 만든다 ('abc'[::-1] → 'cba')
expected_outputs = [s[::-1] for s in test_data]

# 모델 예측 및 정확도 계산
# 문자열 전체가 한 글자도 틀리지 않고 맞아야 "맞음"으로 센다 (위치별 정확도보다 엄격한 기준)
correct_predictions = 0
total_predictions = len(test_data)

# 20개를 하나씩 예측한다 (한 번에 묶어서 predict해도 되지만, 한 개씩 결과를 출력해 보려는 구성)
for i, test_string in enumerate(test_data):
    # 입력 문자열 → 번호 리스트 (학습 때와 같은 변환을 해 줘야 함)
    numbers = [char_to_index[char] for char in test_string]
    array_numbers = np.array([numbers])
    padded_numbers = pad_sequences(array_numbers, maxlen=cal_len)
    # prediction shape : (1, 5, 52) = 샘플 1개, 5개 위치, 위치별 52종류 확률
    prediction = model.predict(padded_numbers)
    # argmax(axis=-1) : 위치마다 확률이 가장 높은 번호를 고른다 → 길이 5의 번호 배열. [0]은 샘플 1개를 꺼냄
    predicted_indices = np.argmax(prediction, axis=-1)[0]
    # 번호 → 문자(index_to_char) → 하나의 문자열로 합친다 ("".join)
    predicted_chars = [index_to_char[idx] for idx in predicted_indices]
    predicted_output = "".join(predicted_chars)
    
    # 문자열이 완전히 같을 때만 True
    is_correct = predicted_output == expected_outputs[i]
    if is_correct:
        correct_predictions += 1
    print(f"입력: {test_string}")
    print(f"예측된 출력: {predicted_output}")
    print(f"실제 출력: {expected_outputs[i]}")
    print(f"정확 여부: {'맞음' if is_correct else '틀림'}\n")

# 총 정확도 = 문자열 전체가 맞은 개수 / 20. 20개뿐이라 5% 단위로만 나오고, 실행할 때마다 값이 달라질 수 있다
accuracy = correct_predictions / total_predictions
print(f"총 정확도: {accuracy * 100:.2f}%")


[['i' 'k' 'M' 'I' 'Z']
 ['m' 'i' 'L' 'U' 'e']
 ['h' 'K' 'r' 'D' 'B']
 ['v' 'Q' 'z' 'P' 'j']]
[['Z' 'I' 'M' 'k' 'i']
 ['e' 'U' 'L' 'i' 'm']
 ['B' 'D' 'r' 'K' 'h']
 ['j' 'P' 'z' 'Q' 'v']]


C:\Users\fasof\.conda\envs\tf219\lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 5, 100)              │           5,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ (None, 5, 128)              │          84,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 5, 52)               │           6,708 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 96,388 (376.52 KB)

 Trainable params: 96,388 (376.52 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.1659 - loss: 3.5989 - val_accuracy: 0.4414 - val_loss: 1.8158
Epoch 2/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4602 - loss: 1.6416 - val_accuracy: 0.4802 - val_loss: 1.4420
Epoch 3/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5004 - loss: 1.3469 - val_accuracy: 0.5076 - val_loss: 1.2945
Epoch 4/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5296 - loss: 1.2089 - val_accuracy: 0.5226 - val_loss: 1.2100
Epoch 5/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5614 - loss: 1.1047 - val_accuracy: 0.5494 - val_loss: 1.1374
Epoch 6/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5881 - loss: 1.0303 - val_accuracy: 0.5702 - val_loss: 1.0862
Epoch 7/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6174 - loss: 0.9611 - val_accuracy: 0.5940 - val_loss: 1.0281
Epoch 8/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6460 - loss: 0.8982 - val_accu